# 01 — Dataset Creation (Register Bias in Arabic LLM-as-Judge)

**الهدف:** من EDRAC (Egyptian فقط) نطلّع benchmark فيه 30 سؤال، ولكل سؤال 3 مستويات جودة × 4 registers.

| Level | Meaning | Required? |
|---|---|---|
| `correct` | الإجابة الذهبية من EDRAC | نعم (120 item) |
| `partial` | جزء صح، ناقص عنصر أساسي | extension |
| `incorrect` | سليمة لغويًا لكن غلط في المحتوى | extension |

**Pipeline:** فلترة الأسئلة ← توليد partial/incorrect بالمصري ← تحويل كل إجابة لـ MSA / Franco / Code-switching ← تحقق تلقائي من التكافؤ ← ملفات الـ annotation.

**مبدأ التصميم:** كل الـ registers بتتولّد من *نفس* الإجابة المصرية، فالغلط (في partial/incorrect) بيتنقل زي ما هو، والفرق الوحيد هو طريقة الكتابة.

**قبل التشغيل:** شغّلي الـ cells بالترتيب من فوق. المفتاح بيتقرا من Colab Secrets (`GEMINI_API_KEY`) أو بيتطلب منك مرة واحدة في الـ cell الخاصة بالاختبار. مفيش cells ترقيع.

In [ ]:
%pip install -q datasets pandas google-genai openai scikit-learn scipy tqdm

import os, re, json, hashlib, random, time
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ------------------------- Config -------------------------
SEED = 42
PROVIDER = "gemini"                # "gemini" or "openai"
GEN_MODEL = "gemini-3.8-flash"     # writes partial/incorrect + register rewrites
CHECK_MODEL = "gemini-3.7-flash"   # equivalence checker: a DIFFERENT model than GEN_MODEL on purpose
FALLBACK_MODELS = ["gemini-3.6-flash", "gemini-3.5-flash", "gemini-flash-latest", "gemini-3.5-flash-lite"]   # tried in order if a model is busy or out of quota; [] disables
MAX_RPM = 8                        # client-side cap on requests per minute (lower it to 4-5 if you still see 429)
N_QUESTIONS = 30
N_RESERVE = 6                      # extra questions in case some are dropped after manual review
ANNOTATE_LEVELS = ["correct", "partial", "incorrect"]   # use ["correct", "incorrect"] if time is short
DATA = Path("data"); DATA.mkdir(exist_ok=True)
CACHE_PATH = DATA / "llm_cache.jsonl"

random.seed(SEED); np.random.seed(SEED)

def get_key(name):
    # order: environment -> Colab Secrets -> ask once (hidden input). No separate "fix" cell needed.
    key = os.environ.get(name)
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get(name)
        except Exception:
            key = None
    if not key:
        import getpass
        key = getpass.getpass(f"{name}: ")
    os.environ[name] = key
    return key

In [ ]:
# ------------------- LLM wrapper (cached + retried) -------------------
# Every response is cached in data/llm_cache.jsonl, so re-running the notebook is free,
# survives Colab disconnects, and gives identical outputs (reproducibility).

def parse_json(text, required=()):
    # required: tuple of keys, or dict {key: (subkeys...)} to also validate nested keys
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text).strip()
    obj = json.loads(text)
    for k in required:
        if k not in obj:
            raise ValueError(f"missing key: {k}")
        subs = required[k] if isinstance(required, dict) else ()
        for sk in subs:
            if not isinstance(obj[k], dict) or sk not in obj[k]:
                raise ValueError(f"missing key: {k}.{sk}")
    return obj

_cache = {}
if CACHE_PATH.exists():
    for line in CACHE_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line); _cache[r["key"]] = r["text"]

_clients = {}
def _client():
    if PROVIDER not in _clients:
        if PROVIDER == "gemini":
            from google import genai
            _clients[PROVIDER] = genai.Client(api_key=get_key("GEMINI_API_KEY"))
        else:
            from openai import OpenAI
            _clients[PROVIDER] = OpenAI(api_key=get_key("OPENAI_API_KEY"))
    return _clients[PROVIDER]

def _raw_call(prompt, model, temperature):
    c = _client()
    if PROVIDER == "gemini":
        from google.genai import types
        r = c.models.generate_content(
            model=model, contents=prompt,
            config=types.GenerateContentConfig(temperature=temperature,
                                               response_mime_type="application/json"))
        return r.text
    r = c.chat.completions.create(
        model=model, temperature=temperature,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

_FATAL = ("API key", "NOT_FOUND", "PERMISSION_DENIED", "UNAUTHENTICATED", "no longer available", "INVALID_ARGUMENT")
_TRANSIENT = ("429", "500", "502", "503", "504", "RESOURCE_EXHAUSTED", "UNAVAILABLE", "DEADLINE_EXCEEDED",
              "overloaded", "high demand")
_down_until = {}     # model -> timestamp; a model that just failed is skipped for a while (circuit breaker)
USAGE = {}           # model -> number of responses used (report this in the paper if a fallback was needed)
_last_call = [0.0]

def _throttle():
    # stay under the free-tier requests-per-minute limit
    wait = _last_call[0] + 60.0 / MAX_RPM - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()

def _key(prompt, model, temperature):
    return hashlib.sha256(f"{PROVIDER}|{model}|{temperature}|{prompt}".encode("utf-8")).hexdigest()

def _cache_get(prompt, model, temperature, required):
    k = _key(prompt, model, temperature)
    if k in _cache:
        try:
            return parse_json(_cache[k], required)
        except Exception:
            return None
    return None

def _attempt(prompt, model, temperature, required, retries):
    last = None
    for attempt in range(retries):
        try:
            _throttle()
            text = _raw_call(prompt, model, temperature)
            obj = parse_json(text, required)
            k = _key(prompt, model, temperature)
            _cache[k] = text
            with open(CACHE_PATH, "a", encoding="utf-8") as f:
                f.write(json.dumps({"key": k, "model": model, "text": text}, ensure_ascii=False) + "\n")
            return obj
        except Exception as e:
            last = e
            msg = str(e)
            if any(m in msg for m in _FATAL):
                raise RuntimeError(f"Fatal API error (not retrying): {msg[:300]}") from e
            quota = ("429" in msg) or ("RESOURCE_EXHAUSTED" in msg)
            if quota and re.search(r"PerDay|per day|daily", msg, re.I):
                raise RuntimeError(f"Daily quota exhausted for {model}: {msg[:200]}") from e
            if attempt < retries - 1:
                transient = any(m in msg for m in _TRANSIENT)
                if quota:                                   # use the server's own retry delay when it gives one
                    d = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", msg) or re.search(r"retry in (\d+(?:\.\d+)?)s", msg)
                    wait = min(90, float(d.group(1)) + 2) if d else 30 * (attempt + 1)
                else:
                    wait = min(120, 10 * 2 ** attempt) if transient else 2 ** attempt
                if transient:
                    print(f"[retry {attempt + 1}/{retries - 1}] {model}: {'quota/rate limit' if quota else 'server busy'}, waiting {wait:.0f}s")
                time.sleep(wait)
    raise RuntimeError(f"LLM call failed after {retries} tries: {last}")

def call_json(prompt, model, temperature=0.0, required=(), retries=4):
    chain = [model] + [m for m in FALLBACK_MODELS if m != model]
    for m in chain:                                    # 1) any cached answer (even from a fallback) is reused
        hit = _cache_get(prompt, m, temperature, required)
        if hit is not None:
            USAGE[m] = USAGE.get(m, 0) + 1
            return hit
    now = time.time()
    todo = [m for m in chain if now >= _down_until.get(m, 0)] or chain    # 2) skip models that just failed
    errors = []
    for m in todo:
        try:
            obj = _attempt(prompt, m, temperature, required, retries if m == model else 3)
            USAGE[m] = USAGE.get(m, 0) + 1
            if m != model:
                print(f"[fallback] answered by {m} (requested {model})")
            return obj
        except RuntimeError as e:
            errors.append(f"{m}: {str(e)[:150]}")
            em = str(e)
            _down_until[m] = time.time() + (6 * 3600 if em.startswith("Daily") else 3600 if em.startswith("Fatal") else 300)
    hint = ""
    if errors and all("Daily quota" in x for x in errors):
        hint = ("\n\nDaily free quota is exhausted on every model. Nothing is lost: all finished answers are in data/llm_cache.jsonl. "
                "Re-run this cell after the quota resets (usually next day), or use another API key / enable billing.")
    raise RuntimeError("All models failed:\n  " + "\n  ".join(errors) + hint)

def fill(template, **kw):
    for k, v in kw.items():
        template = template.replace(f"<<{k}>>", str(v))
    return template

In [ ]:
# Connectivity test: must print {'ok': True} for both models. If it fails, the error message says exactly why.
for m in (GEN_MODEL, CHECK_MODEL):
    print(m, "->", call_json('Return JSON: {"ok": true}', m))

In [ ]:
# OPTIONAL: list the text models available to your key (to edit GEN_MODEL / CHECK_MODEL / FALLBACK_MODELS in the config cell).
if PROVIDER == "gemini":
    bad = ("live", "audio", "tts", "image", "native", "embed", "robotics", "lyria", "veo", "imagen")
    print([m.name.replace("models/", "") for m in _client().models.list()
           if "flash" in m.name and not any(b in m.name for b in bad)])

## 1) Load EDRAC (Egyptian only)

In [ ]:
from huggingface_hub import login
from datasets import load_dataset

login()   # paste your Hugging Face token
raw = load_dataset("MBZUAI/EDRAC", split="test").to_pandas()
raw = raw[raw["Dialect"] == "egy"].reset_index(drop=True)

def clean_passage(p):
    p = re.sub(r"\*\*[^*]*\*\*", " ", str(p))      # remove "** 0 المتحدث **" speaker headers
    return re.sub(r"\s+", " ", p).strip()

records = []
for _, row in raw.iterrows():
    for i in range(1, 11):
        q, a = row[f"Question {i}"], row[f"Answer {i}"]
        if pd.isna(q) or pd.isna(a) or not str(q).strip() or not str(a).strip():
            continue
        records.append({"source_id": row["ID"], "passage": clean_passage(row["Passage"]),
                        "qa_index": i, "question": str(q).strip(), "answer": str(a).strip()})
qa = pd.DataFrame(records)
print("Egyptian QA pairs:", len(qa), "| passages:", qa.source_id.nunique())

## 2) Filter + sample questions

الفلتر بيشيل: yes/no، إجابات قصيرة أو طويلة جدًا، أسئلة بتشاور على الـ passage ("المتحدث"، "الفيديو"…)، وإجابات مش عربي. القائمة قابلة للتعديل.

In [ ]:
AR_CHAR = re.compile(r"[\u0600-\u06FF]")
def arabic_ratio(s):
    letters = [c for c in s if c.isalpha()]
    return sum(bool(AR_CHAR.match(c)) for c in letters) / max(len(letters), 1)

def norm_ar(s):
    s = re.sub(r"[\u064B-\u0652\u0640]", "", s)
    return s.translate(str.maketrans("أإآ", "ااا")).strip()

DEICTIC = re.compile(r"المتحدث|المتحدثة|الفيديو|الحلقة|المقطع|النص|الكلام ده|المذيع|الضيف|الراوي")
YESNO = {norm_ar(x) for x in ["ايوه", "أيوه", "اه", "آه", "أه", "لا", "لأ", "لاء", "نعم", "ايوة"]}

f = qa.copy()
f["q_words"] = f.question.str.split().str.len()
f["a_words"] = f.answer.str.split().str.len()
ok = f.a_words.between(3, 25) & f.q_words.between(4, 25)
ok &= f.answer.map(arabic_ratio) >= 0.8
ok &= ~f.question.map(lambda q: bool(DEICTIC.search(q)))
ok &= ~f.question.map(lambda q: norm_ar(q).startswith("هل") or " ولا " in f" {q} ")
ok &= ~f.answer.map(lambda a: norm_ar(a).strip(" .،!؟") in YESNO)
pool = f[ok]
print(f"Passed filter: {len(pool)} / {len(f)} pairs | passages: {pool.source_id.nunique()}")

need = N_QUESTIONS + N_RESERVE
questions = (pool.sample(frac=1, random_state=SEED)
                 .drop_duplicates("source_id")      # one question per passage
                 .head(need).reset_index(drop=True))
assert len(questions) == need, "Filter too strict: relax the filters above"
questions.insert(0, "question_id", [f"Q{i:03d}" for i in range(1, need + 1)])
questions["status"] = ["selected"] * N_QUESTIONS + ["reserve"] * N_RESERVE

# Manual review sheet: set keep=0 for any bad question, save as data/question_review_done.csv, re-run from here.
review = questions[["question_id", "status", "source_id", "question", "answer"]].copy()
review["passage_excerpt"] = questions.passage.str[:400]
review["keep"] = 1; review["note"] = ""
review.to_csv(DATA / "question_review.csv", index=False, encoding="utf-8-sig")

done = DATA / "question_review_done.csv"
if done.exists():
    keep_ids = pd.read_csv(done).query("keep == 1")["question_id"].tolist()
    questions = questions[questions.question_id.isin(keep_ids)].reset_index(drop=True)
    questions["status"] = ["selected" if i < N_QUESTIONS else "reserve" for i in range(len(questions))]
    print("Applied manual review ->", (questions.status == "selected").sum(), "selected questions")

questions[["question_id", "status", "source_id", "question", "answer"]].head(10)

## 3) Generate `partial` and `incorrect` answers (Egyptian)

In [ ]:
PROMPT_LEVELS = """You are helping build a benchmark for evaluating answers to questions about a spoken Egyptian Arabic passage.

Passage (noisy transcript, may be truncated):
<<passage>>

Question: <<question>>
Gold answer (Egyptian Arabic): <<answer>>

Write two NEW candidate answers in natural Egyptian Arabic (Arabic script), with similar length and style to the gold answer:
1. "partial": contains part of the correct information but misses a key element or is too vague to fully answer. It must NOT contain any false statement.
2. "incorrect": fluent and plausible, but factually WRONG with respect to the passage (for example a wrong entity, number, or reason). It must not be absurd and must not contradict itself.

Return JSON only: {"partial": "...", "incorrect": "..."}"""

egy_levels = {}   # question_id -> {"correct","partial","incorrect"}
for _, r in tqdm(questions.iterrows(), total=len(questions), desc="levels"):
    p = fill(PROMPT_LEVELS, passage=r.passage[:3000], question=r.question, answer=r.answer)
    out = call_json(p, GEN_MODEL, temperature=0.7, required=("partial", "incorrect"))
    egy_levels[r.question_id] = {"correct": r.answer,
                                 "partial": out["partial"].strip(),
                                 "incorrect": out["incorrect"].strip()}
pd.DataFrame(egy_levels).T.head(5)

## 4) Convert every Egyptian answer to MSA / Franco / Code-switching

مكالمة واحدة لكل سؤال (بتحوّل المستويات التلاتة مع بعض) بدل 3 مكالمات، عشان نقلّل استهلاك الـ quota.

In [ ]:
PROMPT_REGISTERS = """Rewrite each of the three Egyptian Arabic answers below into three other writing registers WITHOUT changing its meaning. Do not add, remove, correct, or soften any information. If an answer is wrong or incomplete, keep it exactly as wrong or incomplete. Keep all names, numbers, and quantities identical. Rewrite each answer independently.

Question (for disambiguation only): <<question>>

Egyptian answers:
correct: <<correct>>
partial: <<partial>>
incorrect: <<incorrect>>

Registers:
- "msa": Modern Standard Arabic, Arabic script, formal grammar.
- "franco": Egyptian Arabic written in Latin letters (Arabizi) with the usual digit conventions: 2=ء 3=ع 5=خ 7=ح 8=غ 9=ق 6=ط. No Arabic script at all.
- "codeswitch": Egyptian Arabic in Arabic script where 1-3 content words or terms (e.g. a named entity or technical term) are written in English Latin letters, as an Egyptian speaker naturally mixes. Arabic grammar and sentence structure stay intact.

Example for "القاهرة عاصمة مصر": msa="القاهرة هي عاصمة جمهورية مصر العربية.", franco="El Qahera 3asmet Masr.", codeswitch="Cairo هي عاصمة مصر."

Return JSON only:
{"correct": {"msa": "...", "franco": "...", "codeswitch": "..."}, "partial": {"msa": "...", "franco": "...", "codeswitch": "..."}, "incorrect": {"msa": "...", "franco": "...", "codeswitch": "..."}}"""

LEVELS = ["correct", "partial", "incorrect"]
REG3 = ("msa", "franco", "codeswitch")
REGS = ["msa", "egyptian", "franco", "codeswitch"]
versions = {}     # (question_id, level) -> {"msa","egyptian","franco","codeswitch"}
qtext = questions.set_index("question_id").question.to_dict()

for qid, lv in tqdm(egy_levels.items(), desc="registers (1 call per question)"):
    out = call_json(fill(PROMPT_REGISTERS, question=qtext[qid], correct=lv["correct"],
                         partial=lv["partial"], incorrect=lv["incorrect"]),
                    GEN_MODEL, temperature=0.2, required={l: REG3 for l in LEVELS})
    for lvl in LEVELS:
        versions[(qid, lvl)] = {"egyptian": lv[lvl], **{k: str(out[lvl][k]).strip() for k in REG3}}
len(versions)

## 5) Automatic checks: rules + LLM meaning-equivalence

الفحص الآلي بيكتشف المشاكل الواضحة فقط. **المراجعة البشرية هي المرجع النهائي** (الـ equivalence sheet في الخطوة 7).

In [ ]:
def digits_of(s):
    s = s.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    return sorted(re.findall(r"\d+", s))

LATIN_WORD = re.compile(r"[A-Za-z]{2,}")
def rule_check(reg, text, egy):
    if reg == "egyptian":
        return arabic_ratio(text) >= 0.8
    if reg == "msa":
        return arabic_ratio(text) >= 0.9 and digits_of(text) == digits_of(egy)
    if reg == "franco":                       # digits are letters in Franco, so no digit check here
        return not AR_CHAR.search(text) and bool(LATIN_WORD.search(text))
    if reg == "codeswitch":
        return bool(AR_CHAR.search(text)) and bool(LATIN_WORD.search(text)) and digits_of(text) == digits_of(egy)

PROMPT_CHECK = """You are a strict Arabic meaning-equivalence checker.

Below, for each quality level, there is a source answer in Egyptian Arabic and three rewritten versions. For EACH version decide whether it conveys EXACTLY the same information as its source: nothing added, nothing missing, nothing changed (names, numbers, polarity, certainty). IGNORE differences in dialect, script, spelling, and English loanwords. Errors or omissions that are already in the source must be preserved, not counted as differences.

<<blocks>>

Return JSON only, in this shape (one entry per level and per version):
{"correct": {"msa": {"same_meaning": true, "issue": ""}, "franco": {"same_meaning": true, "issue": ""}, "codeswitch": {"same_meaning": true, "issue": ""}}, "partial": {...same shape...}, "incorrect": {...same shape...}}"""

def check_blocks(qid):
    parts = []
    for lvl in LEVELS:
        v = versions[(qid, lvl)]
        parts.append(f"[{lvl}]\nsource: {v['egyptian']}\nmsa: {v['msa']}\nfranco: {v['franco']}\ncodeswitch: {v['codeswitch']}")
    return "\n\n".join(parts)

auto_check = {}   # (qid, level, register) -> (same_meaning, issue)
for qid in tqdm(egy_levels, desc="equivalence check (1 call per question)"):
    out = call_json(fill(PROMPT_CHECK, blocks=check_blocks(qid)), CHECK_MODEL, temperature=0.0,
                    required={l: REG3 for l in LEVELS})
    for lvl in LEVELS:
        for reg in REG3:
            r = out[lvl][reg] if isinstance(out[lvl][reg], dict) else {}
            auto_check[(qid, lvl, reg)] = (bool(r.get("same_meaning")), str(r.get("issue", "")))
len(auto_check)

## 6) Assemble the long-format dataset

In [ ]:
LVL_CODE = {"correct": "C", "partial": "P", "incorrect": "I"}
REG_CODE = {"msa": "MSA", "egyptian": "EGY", "franco": "FRA", "codeswitch": "CSW"}

rows = []
meta = questions.set_index("question_id")
for (qid, lvl), v in versions.items():
    for reg in REGS:
        same, issue = (True, "") if reg == "egyptian" else auto_check[(qid, lvl, reg)]
        rows.append({
            "item_id": f"{qid}-{LVL_CODE[lvl]}-{REG_CODE[reg]}",
            "question_id": qid, "status": meta.loc[qid, "status"],
            "source_id": meta.loc[qid, "source_id"], "question": meta.loc[qid, "question"],
            "reference_egyptian": egy_levels[qid]["correct"],
            "level": lvl, "register": reg, "core": lvl == "correct",
            "answer": v[reg], "egyptian_source": v["egyptian"],
            "rule_ok": rule_check(reg, v[reg], v["egyptian"]),
            "auto_same_meaning": same, "auto_issue": issue,
        })
cand = pd.DataFrame(rows).sort_values(["question_id", "level", "register"]).reset_index(drop=True)
cand.to_csv(DATA / "candidates_long.csv", index=False, encoding="utf-8-sig")

print("Items:", len(cand), "| core (correct):", int(cand.core.sum()))
print("\nFlagged by rules:", int((~cand.rule_ok).sum()), "| flagged by LLM check:", int((~cand.auto_same_meaning).sum()))
cand[~(cand.rule_ok & cand.auto_same_meaning)][["item_id", "register", "answer", "auto_issue"]]

## 7) Files for the human annotators

- `annotator_1.csv` / `annotator_2.csv`: **ترتيب عشوائي مختلف** لكل annotator، والعمود `rating` فاضي.
- `equivalence_check.csv`: بيتراجع فيه التكافؤ (same_meaning + نسخة مصحّحة لو لزم). **راجعيه قبل تدوير الـ ratings** لأن التصحيح بيغيّر النص.
- `rubric.md`: مقياس التقييم (المحتوى فقط).

In [ ]:
RUBRIC = """# Rating rubric (content correctness ONLY)

Rate how well the candidate answer answers the question, compared with the reference answer.
IGNORE dialect, script (Arabic/Latin), spelling, grammar, and English words. Judge only the information.

5 = Fully correct and complete
4 = Correct, with a minor omission
3 = Partially correct (some key information missing, nothing false)
2 = Mostly wrong, with a trace of correct information
1 = Wrong or irrelevant

Do not try to guess which writing style is "better". Two answers with the same information must get the same rating.
"""
(DATA / "rubric.md").write_text(RUBRIC, encoding="utf-8")

sel = cand[(cand.status == "selected") & cand.level.isin(ANNOTATE_LEVELS)]
for k, seed in [(1, SEED), (2, SEED + 1)]:
    sheet = (sel.sample(frac=1, random_state=seed)
                [["item_id", "question", "reference_egyptian", "answer"]]
                .rename(columns={"answer": "candidate_answer"}))
    sheet["rating"] = ""; sheet["notes"] = ""
    sheet.to_csv(DATA / f"annotator_{k}.csv", index=False, encoding="utf-8-sig")

eq = cand[(cand.status == "selected") & (cand.register != "egyptian")]
eq = eq[["item_id", "register", "egyptian_source", "answer", "auto_same_meaning", "auto_issue"]].copy()
eq["human_same_meaning"] = ""; eq["corrected_answer"] = ""
eq.to_csv(DATA / "equivalence_check.csv", index=False, encoding="utf-8-sig")

print("Rating items per annotator:", len(sel), "| equivalence items:", len(eq))

## 8) Sanity statistics (confounds to report)

In [ ]:
s = cand[cand.status == "selected"].copy()
s["n_words"] = s.answer.str.split().str.len()
s["n_chars"] = s.answer.str.len()
print(s.groupby("register")[["n_words", "n_chars"]].mean().round(1))
print("\nItems per level x register:")
print(pd.crosstab(s.level, s.register))
print("\nLLM responses used per model (mention in the paper if a fallback was used):", USAGE)
print("\nAuto equivalence pass rate per register:")
print(s[s.register != "egyptian"].groupby("register")[["rule_ok", "auto_same_meaning"]].mean().round(3))

## 9) بعد ما الـ annotators يخلّصوا (شغّلي من هنا)

احفظي الملفات المعبّأة باسم `data/annotator_1_done.csv` و `data/annotator_2_done.csv`.

In [ ]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

a1 = pd.read_csv(DATA / "annotator_1_done.csv")[["item_id", "rating"]]
a2 = pd.read_csv(DATA / "annotator_2_done.csv")[["item_id", "rating"]]
m = a1.merge(a2, on="item_id", suffixes=("_1", "_2")).dropna()
m[["rating_1", "rating_2"]] = m[["rating_1", "rating_2"]].astype(int)

kappa = cohen_kappa_score(m.rating_1, m.rating_2, weights="quadratic")
rho = spearmanr(m.rating_1, m.rating_2)[0]
exact = (m.rating_1 == m.rating_2).mean()
within1 = ((m.rating_1 - m.rating_2).abs() <= 1).mean()
print(f"N={len(m)} | weighted kappa={kappa:.3f} | Spearman={rho:.3f} | exact={exact:.1%} | within-1={within1:.1%}")

m["human_mean"] = (m.rating_1 + m.rating_2) / 2
human = cand.merge(m[["item_id", "rating_1", "rating_2", "human_mean"]], on="item_id")
human.to_csv(DATA / "human_ratings.csv", index=False, encoding="utf-8-sig")
print("Saved data/human_ratings.csv:", len(human), "rows")